# BraTS 2021 Preprocessing Pipeline — All Fixes Applied
**SEIS 766 Vision AI — Spring 2026**

Corrected task ordering: `SEG-01 → SEG-02 → SEG-03 → SEG-06 → SEG-04 → SEG-05`

All fixes from code review applied:
1. SEG-01: Spot-check 3 random cases with shape verification
2. SEG-03: `np.nan_to_num()` before normalization
3. SEG-06: `.copy()` before shuffle to protect caller's list
4. SEG-06: Check if split JSON already exists
5. SEG-04: `+1` on end_z to include full margin
6. SEG-04: Partition-aware extraction (train/val saved separately)
7. SEG-05: `.copy()` after `np.flip` for array contiguity
8. SEG-05: Geometric and photometric transforms separated
9. SEG-05: `borderMode`/`borderValue` in warpAffine
10. SEG-05: Mask cast to float32 before warp, uint8 after
11. SEG-05: Wrapped in Dataset class (train augments, val doesn't)
12. Execution: leakage verification check


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/"


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SMART ROUTER: Detect which scenario applies and skip work
#
# This cell inspects Drive + local SSD to figure out the fastest
# path for your current situation, then tells you exactly which
# cells to run.
#
# Scenario A (first time ever, ~25-40 min):
#   - brats_slices.zip does NOT exist on shared Drive
#   - Run ALL cells below in order
#
# Scenario B (slices already extracted, ~2-5 min):
#   - brats_slices.zip EXISTS on shared Drive
#   - SKIP cells 3-10 entirely
#   - Run Cell 2 (imports), Cell 11 (load slices zip), Cell 12
#     (dataset wrapper), Cell 13 (create datasets)
#
# Scenario C (local slices still exist from prior session, instant):
#   - /content/local_slices/slices/train/ has .npz files
#   - Runtime didn't disconnect — SKIP everything preprocessing
#   - Go straight to Cell 12, Cell 13, then Phase 2
# ═══════════════════════════════════════════════════════════════

import os

# Paths (these get set properly in Cell 2, but we need them now)
_ROOT = '/content/drive/MyDrive/'
_SHARED = _ROOT + 'Group_Project_766'
_SLICES_ZIP = os.path.join(_SHARED, 'brats_slices.zip')
_RAW_ZIP = os.path.join(_SHARED, 'brats_raw.zip')
_LOCAL_SLICES = '/content/local_slices/slices/train'

has_local_slices = os.path.exists(_LOCAL_SLICES) and \
                   len([f for f in os.listdir(_LOCAL_SLICES) if f.endswith('.npz')]) > 100 \
                   if os.path.exists(_LOCAL_SLICES) else False
has_slices_zip = os.path.exists(_SLICES_ZIP)
has_raw_zip = os.path.exists(_RAW_ZIP)

print('═' * 60)
print('ENVIRONMENT CHECK')
print('═' * 60)
print(f'Local extracted slices (this session): {"✅ YES" if has_local_slices else "❌ no"}')
print(f'brats_slices.zip on shared Drive:      {"✅ YES" if has_slices_zip else "❌ no"}')
print(f'brats_raw.zip on shared Drive:         {"✅ YES" if has_raw_zip else "❌ no"}')
print()

if has_local_slices:
    print('🚀 SCENARIO C — Local slices already extracted (instant)')
    print()
    print('   WHAT TO RUN:')
    print('   → Cell 2    (imports + paths)')
    print('   → Cell 12   (SEG-05 augmentation functions)')
    print('   → Cell 13   (create datasets)')
    print('   → Phase 2 cells (starting at Cell 15)')
    print()
    print('   SKIP cells 3, 4, 5, 6, 7, 8, 9, 10, 11 — not needed.')

elif has_slices_zip:
    print('⚡ SCENARIO B — Slices zip on Drive (2-5 min to unzip)')
    print()
    print('   WHAT TO RUN:')
    print('   → Cell 2    (imports + paths)')
    print('   → Cell 11   (unzip brats_slices.zip to local SSD)')
    print('   → Cell 12   (SEG-05 augmentation functions)')
    print('   → Cell 13   (create datasets)')
    print('   → Phase 2 cells (starting at Cell 15)')
    print()
    print('   SKIP cells 3, 4, 5, 6, 7, 8, 9, 10 — slices already done.')

elif has_raw_zip:
    print('🏗️  SCENARIO A — First-time full extraction (25-40 min)')
    print()
    print('   WHAT TO RUN (in order):')
    print('   → Cell 2    (imports + paths)')
    print('   → Cell 3    (unzip raw BraTS from Drive, ~5-10 min)')
    print('   → Cells 4-7 (SEG-01 through SEG-06)')
    print('   → Cell 8    (define parallel extraction functions)')
    print('   → Cell 9    (run parallel extraction, ~15-20 min)')
    print('   → Cell 10   (zip slices + upload to Drive, ~5-10 min)')
    print('   → Cell 12   (SEG-05 augmentation functions)')
    print('   → Cell 13   (create datasets)')
    print('   → Phase 2 cells (starting at Cell 15)')
    print()
    print('   SKIP cell 11 — no slices zip exists yet (you\'ll create it in Cell 10).')

else:
    print('❌ NO DATA FOUND')
    print()
    print('   No raw zip, no slices zip, no local slices.')
    print('   Check that the shared Drive folder is mounted correctly.')
    print(f'   Expected: {_SHARED}')
    print(f'   Exists:   {os.path.exists(_SHARED)}')

print()
print('═' * 60)


In [ ]:
import os
import json
import random
import time
import shutil
import zipfile
import numpy as np
import cv2
import nibabel as nib
from concurrent.futures import ProcessPoolExecutor, as_completed

# ═══════════════════════════════════════════════════════════════
# PATHS
#
# SHARED_FOLDER: The shared Drive folder (from an external account)
#   containing brats_raw.zip and where results go.
# LOCAL_DATA:    Local SSD copy of raw BraTS data (fast read)
# LOCAL_DIR:     Local SSD for extracted slices (fast write)
# OUTPUT_DIR:    Persistent Drive storage for split JSON, model
#                checkpoints, and slice zip.
#
# STRATEGY: Unzip raw BraTS to local SSD ONCE, process there,
# then save only the slice zip + model artifacts back to Drive.
#
# NOTE ON SHARED FOLDER: Because the Group_Project_766 folder is
# shared from another Colab account, we have READ-ONLY access by
# default. Write operations go to YOUR MyDrive, not the shared
# folder. If you need the slice zip written to the SHARED folder
# (so teammates can use it), the owner of the shared folder must
# grant you Editor access in Drive sharing settings.
# ═══════════════════════════════════════════════════════════════

# Path to the shared Drive folder (read-only unless granted Editor)
SHARED_FOLDER = ROOT + "Group_Project_766"

# Source of raw BraTS data (zip in the shared folder)
SHARED_RAW_ZIP = os.path.join(SHARED_FOLDER, "brats_raw.zip")

# Optional: If teammates have already extracted slices and the
# slice zip exists in the shared folder, we can unzip directly
# instead of re-running the raw extraction.
SHARED_SLICES_ZIP = os.path.join(SHARED_FOLDER, "brats_slices.zip")

# Local SSD working directories
LOCAL_DATA = "/content/brats_raw"         # unzipped raw data lives here
LOCAL_DIR  = "/content/local_slices"      # extracted slices live here

# Where OUR outputs go (checkpoints, results, etc.)
# This is in the shared folder if you have Editor access, otherwise
# change to a path in your personal MyDrive.
OUTPUT_DIR = SHARED_FOLDER

# Raw data path (used by SEG-01 through SEG-04 for loading)
TRAIN_PATH = LOCAL_DATA

# Sanity checks
print(f"Shared folder exists:  {os.path.exists(SHARED_FOLDER)}")
print(f"Raw zip exists:        {os.path.exists(SHARED_RAW_ZIP)}")
print(f"Slices zip exists:     {os.path.exists(SHARED_SLICES_ZIP)}")

# Check write access to OUTPUT_DIR
try:
    test_file = os.path.join(OUTPUT_DIR, '.write_test.tmp')
    with open(test_file, 'w') as f:
        f.write('test')
    os.remove(test_file)
    print(f"Write access to OUTPUT_DIR: ✅ yes")
except (OSError, PermissionError) as e:
    print(f"Write access to OUTPUT_DIR: ❌ NO ({e})")
    print(f"   Change OUTPUT_DIR to your personal MyDrive path.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Unzip Raw BraTS Data from Shared Drive → Local SSD
#
# SMART LOGIC:
# 1. If slices zip exists on Drive, skip raw data entirely
#    (run the 'Load Slices from Drive Zip' cell instead).
# 2. If local raw data already exists, skip unzip.
# 3. Otherwise, unzip from the shared Drive folder.
#
# HANDLES NESTED ZIP STRUCTURE:
# Some zips contain BraTS2021_* directly at root, others contain
# an intermediate folder like 'brats_raw/BraTS2021_*'. This cell
# detects both and points TRAIN_PATH to the correct level.
# ═══════════════════════════════════════════════════════════════

def find_brats_root(search_dir):
    """Find the directory that directly contains BraTS2021_* folders."""
    for root, dirs, _ in os.walk(search_dir):
        brats_dirs = [d for d in dirs if d.startswith('BraTS2021')]
        if len(brats_dirs) > 100:  # need substantial count to confirm
            return root
    return None

# Check if we can skip raw data entirely (slices already exist)
if os.path.exists(SHARED_SLICES_ZIP):
    print(f"ℹ️  Slices zip detected at {SHARED_SLICES_ZIP}")
    print(f"   You can SKIP this cell and SKIP extraction cells.")
    print(f"   Run the 'Load Slices from Drive Zip' cell instead.")

# Check if local raw data already exists
existing_root = find_brats_root(LOCAL_DATA) if os.path.exists(LOCAL_DATA) else None

if existing_root:
    count = len([d for d in os.listdir(existing_root) if d.startswith('BraTS2021')])
    print(f"✅ Local raw data already exists: {count} cases in {existing_root}")
    TRAIN_PATH = existing_root  # update in case it's nested
elif not os.path.exists(SHARED_RAW_ZIP):
    print(f"❌ Raw zip not found at {SHARED_RAW_ZIP}")
    print(f"   Cannot proceed without raw data or slice zip.")
else:
    print(f"📥 Unzipping raw BraTS data from shared Drive...")
    print(f"   From: {SHARED_RAW_ZIP}")
    print(f"   To:   {LOCAL_DATA}")
    print(f"   (Expected: 5-10 minutes)")

    start = time.time()
    os.makedirs(LOCAL_DATA, exist_ok=True)

    # -q = quiet, -o = overwrite existing (important if a previous
    #  partial unzip left fragments). Using !unzip instead of Python's
    # zipfile because it's faster for large archives.
    !unzip -q -o "{SHARED_RAW_ZIP}" -d "{LOCAL_DATA}/"

    elapsed = time.time() - start

    # Detect the correct BraTS root directory (handles nested structure)
    actual_root = find_brats_root(LOCAL_DATA)
    if actual_root is None:
        print(f"❌ Unzip completed but no BraTS2021_* folders found!")
        print(f"   Check zip structure with: !ls {LOCAL_DATA}")
    else:
        TRAIN_PATH = actual_root  # update path to the real root
        count = len([d for d in os.listdir(actual_root) if d.startswith('BraTS2021')])
        print(f"\n✅ Unzipped {count} cases in {elapsed/60:.1f} minutes")
        if actual_root != LOCAL_DATA:
            print(f"   (BraTS root is nested at: {actual_root})")
        print(f"   TRAIN_PATH is now: {TRAIN_PATH}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-01: Verify BraTS 2021 Dataset (reads from local SSD)
# ═══════════════════════════════════════════════════════════════

def verify_data():
    if not os.path.exists(TRAIN_PATH):
        print(f"❌ Folder not found at: {TRAIN_PATH}")
        return []

    cases = sorted([d for d in os.listdir(TRAIN_PATH)
                    if d.startswith('BraTS2021') and
                    os.path.isdir(os.path.join(TRAIN_PATH, d))])
    print(f"📂 Found {len(cases)} training cases in {TRAIN_PATH}")

    if cases:
        spot_checks = random.sample(cases, min(3, len(cases)))
        suffixes = ['_t1.nii.gz','_t1ce.nii.gz','_t2.nii.gz','_flair.nii.gz','_seg.nii.gz']
        for cid in spot_checks:
            for s in suffixes:
                fp = os.path.join(TRAIN_PATH, cid, f"{cid}{s}")
                if not os.path.exists(fp):
                    print(f"  ⚠️ Missing: {fp}")
                else:
                    img = nib.load(fp)
                    ok = '✅' if img.shape == (240,240,155) else '⚠️'
                    print(f"  {ok} {cid}{s} → {img.shape}")
    return cases

all_cases = verify_data()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-02: NIfTI Loading Pipeline
# ═══════════════════════════════════════════════════════════════

def load_case_data(case_id, data_path=None):
    """Load all 4 modalities + seg mask for one case."""
    if data_path is None:
        data_path = TRAIN_PATH
    case_path = os.path.join(data_path, case_id)
    data_dict = {}
    for mod in ['t1', 't1ce', 't2', 'flair']:
        fp = os.path.join(case_path, f"{case_id}_{mod}.nii.gz")
        data_dict[mod] = nib.load(fp).get_fdata().astype(np.float32)
    seg_fp = os.path.join(case_path, f"{case_id}_seg.nii.gz")
    data_dict['seg'] = nib.load(seg_fp).get_fdata().astype(np.uint8)
    return data_dict

if all_cases:
    s = load_case_data(all_cases[0])
    print(f"✅ Loaded {all_cases[0]}: " +
          ', '.join(f"{k}={v.shape}" for k,v in s.items()))


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-03: Per-Volume Z-Score Normalization
# FIX: np.nan_to_num() before computing statistics
# ═══════════════════════════════════════════════════════════════

def normalize_volume(volume):
    volume = np.nan_to_num(volume, nan=0.0, posinf=0.0, neginf=0.0)
    brain_mask = volume > 0
    if np.any(brain_mask):
        mean = volume[brain_mask].mean()
        std = volume[brain_mask].std()
        normalized = (volume - mean) / (std + 1e-8)
        normalized[~brain_mask] = 0
        return normalized.astype(np.float32)
    return volume.astype(np.float32)

def normalize_case(data_dict):
    for mod in ['t1', 't1ce', 't2', 'flair']:
        data_dict[mod] = normalize_volume(data_dict[mod])
    return data_dict

if all_cases:
    norm = normalize_volume(s['flair'])
    t = norm[norm != 0]
    print(f"📊 Normalization: mean={t.mean():.4f}, std={t.std():.4f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-06: Case-Level Train/Val Split
# FIX: .copy() before shuffle
# FIX: Check if split JSON already exists
# Split JSON saves to Drive (persistent across sessions)
# ═══════════════════════════════════════════════════════════════

def save_dataset_split(case_ids, train_ratio=0.8):
    random.seed(42)
    shuffled = case_ids.copy()
    random.shuffle(shuffled)
    split_point = int(len(shuffled) * train_ratio)
    split_dict = {'train': shuffled[:split_point], 'val': shuffled[split_point:]}

    output_path = os.path.join(OUTPUT_DIR, 'dataset_split.json')
    try:
        with open(output_path, 'w') as f:
            json.dump(split_dict, f, indent=4)
        print(f"✅ Split saved to: {output_path}")
    except OSError as e:
        print(f"❌ Failed: {e}")
    print(f"   Training: {len(split_dict['train'])}, Validation: {len(split_dict['val'])}")
    return split_dict

split_path = os.path.join(OUTPUT_DIR, 'dataset_split.json')
if os.path.exists(split_path):
    print(f"📄 Loading existing split...")
    with open(split_path) as f:
        dataset_split = json.load(f)
    print(f"   {len(dataset_split['train'])} train / {len(dataset_split['val'])} val")
else:
    dataset_split = save_dataset_split(all_cases)

overlap = set(dataset_split['train']) & set(dataset_split['val'])
print(f"{'❌ LEAKAGE!' if overlap else '✅ No leakage'}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-04: Partition-Aware Slice Extraction — PARALLELIZED
#
# OPTIMIZATION: ProcessPoolExecutor for multi-core processing.
# On A100 High-RAM (8-12 cores): ~15-20 min vs 90+ min sequential.
#
# FIXES FROM PREVIOUS VERSION:
# - Atomic write (.tmp then rename) prevents partial files if killed
# - Error messages captured and returned, not silently swallowed
# - Uses TRAIN_PATH (which may have been updated for nested zips)
# ═══════════════════════════════════════════════════════════════

def process_single_case(args):
    """Load → normalize → extract → save. Runs in a separate process.
    Must be a top-level function (not a closure) for pickling."""
    case_id, data_path, save_dir = args

    try:
        # Load all 5 NIfTI files
        case_path = os.path.join(data_path, case_id)
        data_dict = {}
        for mod in ['t1', 't1ce', 't2', 'flair']:
            fp = os.path.join(case_path, f"{case_id}_{mod}.nii.gz")
            data_dict[mod] = nib.load(fp).get_fdata().astype(np.float32)
        seg_fp = os.path.join(case_path, f"{case_id}_seg.nii.gz")
        data_dict['seg'] = nib.load(seg_fp).get_fdata().astype(np.uint8)

        # Per-volume z-score normalization (brain mask only)
        for mod in ['t1', 't1ce', 't2', 'flair']:
            vol = np.nan_to_num(data_dict[mod], nan=0.0, posinf=0.0, neginf=0.0)
            brain = vol > 0
            if np.any(brain):
                m, s = vol[brain].mean(), vol[brain].std()
                vol = (vol - m) / (s + 1e-8)
                vol[~brain] = 0
            data_dict[mod] = vol.astype(np.float32)

        # Find tumor z-range with margin
        seg = data_dict['seg']
        z_idx = np.any(seg > 0, axis=(0, 1))
        if not np.any(z_idx):
            return case_id, 0, None  # no tumor

        start_z = max(0, np.where(z_idx)[0].min() - 5)
        end_z = min(seg.shape[2], np.where(z_idx)[0].max() + 6)

        # Extract and save each slice with atomic write
        count = 0
        for z in range(start_z, end_z):
            img = np.stack([
                data_dict['t1'][:, :, z], data_dict['t1ce'][:, :, z],
                data_dict['t2'][:, :, z], data_dict['flair'][:, :, z]
            ], axis=0)
            msk = seg[:, :, z]

            # Atomic write: save to .tmp, then rename. If process is
            # killed mid-write, no partial .npz exists to crash loaders.
            final_path = os.path.join(save_dir, f"{case_id}_slice_{count:03d}.npz")
            tmp_path = final_path + '.tmp'
            np.savez_compressed(tmp_path, image=img, mask=msk)
            os.rename(tmp_path, final_path)
            count += 1

        return case_id, count, None
    except Exception as e:
        return case_id, -1, str(e)  # return error message


def extract_partition_parallel(case_ids, partition_name, data_path, num_workers=6):
    """Extract slices using multiple CPU cores in parallel."""
    save_dir = os.path.join(LOCAL_DIR, 'slices', partition_name)
    os.makedirs(save_dir, exist_ok=True)

    args_list = [(cid, data_path, save_dir) for cid in case_ids]
    total_slices = 0
    completed = 0
    errors = []
    start_time = time.time()

    print(f"  [{partition_name}] Processing {len(case_ids)} cases "
          f"with {num_workers} parallel workers...")

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(process_single_case, a): a[0]
                   for a in args_list}

        for future in as_completed(futures):
            case_id, count, err = future.result()
            completed += 1
            if count < 0:
                errors.append((case_id, err))
            else:
                total_slices += count

            if completed % 100 == 0 or completed == len(case_ids):
                elapsed = time.time() - start_time
                rate = completed / elapsed * 60 if elapsed > 0 else 0
                print(f"  [{partition_name}] {completed}/{len(case_ids)} cases, "
                      f"{total_slices} slices, {elapsed/60:.1f}min "
                      f"({rate:.0f} cases/min)")

    elapsed = time.time() - start_time
    print(f"✅ {partition_name}: {total_slices} slices in {elapsed/60:.1f} minutes")
    if errors:
        print(f"⚠️  {len(errors)} cases failed:")
        for cid, err in errors[:5]:
            print(f"    {cid}: {err}")
        if len(errors) > 5:
            print(f"    ... and {len(errors)-5} more")

    return total_slices, errors

print("✅ Extraction functions ready (parallelized, atomic, error-reporting)")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Run Parallel Extraction
#
# SKIP THIS CELL if slices already exist (run 'Load Slices from
# Drive Zip' cell instead).
# ═══════════════════════════════════════════════════════════════

import multiprocessing
num_cores = multiprocessing.cpu_count()
NUM_WORKERS = min(8, num_cores - 1)
print(f"💻 Available CPU cores: {num_cores}, using {NUM_WORKERS} workers")

total_start = time.time()

print("\n📤 Extracting training slices...")
train_count, train_errors = extract_partition_parallel(
    dataset_split['train'], 'train', TRAIN_PATH, num_workers=NUM_WORKERS)

print("\n📤 Extracting validation slices...")
val_count, val_errors = extract_partition_parallel(
    dataset_split['val'], 'val', TRAIN_PATH, num_workers=NUM_WORKERS)

total_elapsed = time.time() - total_start
print(f"\n✅ TOTAL: {train_count} train + {val_count} val slices "
      f"in {total_elapsed/60:.1f} minutes")
print(f"   Errors: {len(train_errors)} train, {len(val_errors)} val")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Transfer Slices to Drive (one large zip file)
# ═══════════════════════════════════════════════════════════════

import shutil

zip_path = '/content/brats_slices'
print('📦 Zipping extracted slices...')
shutil.make_archive(zip_path, 'zip', LOCAL_DIR)
print(f'✅ Created {zip_path}.zip')

drive_zip = os.path.join(OUTPUT_DIR, 'brats_slices.zip')
print(f'📤 Copying to Drive...')
shutil.copy2(f'{zip_path}.zip', drive_zip)
print(f'✅ Saved: {drive_zip} ({os.path.getsize(drive_zip)/1e9:.2f} GB)')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Load Slices from Drive Zip (future sessions)
#
# Run THIS cell instead of Cells 3 + 9 + 10 if slices were
# already extracted in a previous session.
# ═══════════════════════════════════════════════════════════════

import zipfile

LOCAL_DIR = '/content/local_slices'
drive_zip = os.path.join(OUTPUT_DIR, 'brats_slices.zip')
local_check = os.path.join(LOCAL_DIR, 'slices', 'train')

if os.path.exists(local_check):
    nt = len([f for f in os.listdir(local_check) if f.endswith('.npz')])
    nv = len([f for f in os.listdir(
        os.path.join(LOCAL_DIR,'slices','val')) if f.endswith('.npz')])
    print(f'✅ Local slices exist: {nt} train + {nv} val')
elif os.path.exists(drive_zip):
    print(f'📥 Extracting from Drive zip...')
    with zipfile.ZipFile(drive_zip, 'r') as z:
        z.extractall(LOCAL_DIR)
    nt = len([f for f in os.listdir(
        os.path.join(LOCAL_DIR,'slices','train')) if f.endswith('.npz')])
    nv = len([f for f in os.listdir(
        os.path.join(LOCAL_DIR,'slices','val')) if f.endswith('.npz')])
    print(f'✅ Unzipped: {nt} train + {nv} val')
else:
    print('⚠️  No zip on Drive. Run extraction cells first.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-05: Train-Only Augmentation
#
# FIX: .copy() after np.flip
# FIX: Geometric (image+mask) separated from photometric (image only)
# FIX: borderMode=BORDER_CONSTANT, borderValue=0.0
# FIX: Mask cast float32 before warp, uint8 after
# FIX: Wrapped in Dataset class with augment flag
# ═══════════════════════════════════════════════════════════════

def geometric_augmentation(image, mask):
    """GEOMETRIC: apply to BOTH image and mask. NN interp on mask."""
    if random.random() > 0.5:
        image = np.flip(image, axis=2).copy()  # FIX: .copy()
        mask = np.flip(mask, axis=1).copy()     # FIX: .copy()

    angle = random.uniform(-15, 15)
    h, w = mask.shape
    rot = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)

    aug_img = np.zeros_like(image)
    for c in range(image.shape[0]):
        aug_img[c] = cv2.warpAffine(
            image[c], rot, (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)

    aug_msk = cv2.warpAffine(
        mask.astype(np.float32), rot, (w, h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT, borderValue=0.0
    ).astype(np.uint8)

    return aug_img, aug_msk


def photometric_augmentation(image):
    """PHOTOMETRIC: IMAGE ONLY. Never apply to mask."""
    image = image * random.uniform(0.9, 1.1)
    if random.random() > 0.5:
        noise = np.random.normal(0, 0.02, image.shape).astype(np.float32)
        image = image + noise
    return image


def apply_augmentation(image, mask):
    """Geometric first (both), then photometric (image only)."""
    image, mask = geometric_augmentation(image, mask)
    image = photometric_augmentation(image)
    return image, mask


class BraTSSliceDataset:
    """Dataset wrapper. augment=True for train, False for val."""
    def __init__(self, slice_dir, augment=False):
        self.slice_dir = slice_dir
        self.augment = augment
        self.file_list = sorted(
            [f for f in os.listdir(slice_dir) if f.endswith('.npz')])
        print(f"📦 {len(self.file_list)} slices "
              f"(augment={'ON' if augment else 'OFF'})")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        data = np.load(os.path.join(self.slice_dir, self.file_list[idx]))
        image = data['image'].astype(np.float32)
        mask = data['mask'].astype(np.uint8)

        if self.augment:
            image, mask = apply_augmentation(image, mask)

        return image.astype(np.float32), mask.astype(np.int64)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Create Datasets and Verify (reads from local SSD)
# ═══════════════════════════════════════════════════════════════

train_dir = os.path.join(LOCAL_DIR, 'slices', 'train')
val_dir = os.path.join(LOCAL_DIR, 'slices', 'val')

train_dataset = BraTSSliceDataset(train_dir, augment=True)
val_dataset = BraTSSliceDataset(val_dir, augment=False)

if len(train_dataset) > 0:
    img_t, msk_t = train_dataset[0]
    print(f'\n🔬 Train: {img_t.shape} {img_t.dtype}, mask labels: {np.unique(msk_t)}')
if len(val_dataset) > 0:
    img_v, msk_v = val_dataset[0]
    print(f'🔬 Val:   {img_v.shape} {img_v.dtype}, mask labels: {np.unique(msk_v)}')
print('\n🚀 Phase 1 complete.')
